In [1]:
"""
Notebook 01: Master Table Construction

Goal: Build genome_gc_env.parquet — one row per genome with
  - genome_id, gtdb_species_clade_id, ncbi_biosample_id
  - gc_pct (CAST from string), genome_size, checkm_completeness, checkm_contamination
  - Pivoted ncbi_env fields: isolation_source, host, env_broad_scale, env_local_scale, env_medium, lat_lon, geo_loc_name
  - has_alphaearth flag

Outputs:
  - projects/gc_ecotype_ecology/data/genome_gc_env.parquet
  - figures/01_gc_distribution_overall.png
  - figures/01_field_coverage.png
"""

'\nNotebook 01: Master Table Construction\n\nGoal: Build genome_gc_env.parquet — one row per genome with\n  - genome_id, gtdb_species_clade_id, ncbi_biosample_id\n  - gc_pct (CAST from string), genome_size, checkm_completeness, checkm_contamination\n  - Pivoted ncbi_env fields: isolation_source, host, env_broad_scale, env_local_scale, env_medium, lat_lon, geo_loc_name\n  - has_alphaearth flag\n\nOutputs:\n  - projects/gc_ecotype_ecology/data/genome_gc_env.parquet\n  - figures/01_gc_distribution_overall.png\n  - figures/01_field_coverage.png\n'

In [2]:
import os
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from berdl_notebook_utils.setup_spark_session import get_spark_session

In [3]:
PROJECT_DIR = "/home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
FIG_DIR = os.path.join(PROJECT_DIR, "figures")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [4]:
spark = get_spark_session()

In [5]:
# -----------------------------------------------------------------------------
# Step 1: Build core genome table with GC, size, quality
# -----------------------------------------------------------------------------
print("Step 1: Building core genome table...")
core = spark.sql("""
    SELECT
      g.genome_id,
      g.gtdb_species_clade_id,
      g.ncbi_biosample_id,
      CAST(m.gc_percentage      AS DOUBLE) AS gc_pct,
      CAST(m.genome_size        AS BIGINT) AS genome_size,
      CAST(m.checkm_completeness AS DOUBLE) AS checkm_completeness,
      CAST(m.checkm_contamination AS DOUBLE) AS checkm_contamination
    FROM kbase_ke_pangenome.genome g
    LEFT JOIN kbase_ke_pangenome.gtdb_metadata m
      ON m.accession = g.genome_id
""")
core.cache()
n_total = core.count()
n_with_gc = core.where(F.col("gc_pct").isNotNull()).count()
print(f"  Total genomes: {n_total:,}")
print(f"  With GC: {n_with_gc:,} ({100*n_with_gc/n_total:.1f}%)")

Step 1: Building core genome table...


  Total genomes: 293,059
  With GC: 293,059 (100.0%)


In [6]:
# -----------------------------------------------------------------------------
# Step 2: Pivot ncbi_env to wide form on the fields we care about
# -----------------------------------------------------------------------------
print("\nStep 2: Pivoting ncbi_env...")
TARGET_FIELDS = [
    "isolation_source",
    "host",
    "env_broad_scale",
    "env_local_scale",
    "env_medium",
    "lat_lon",
    "geo_loc_name",
    "sample_type",
    "host_disease",
]
env_long = spark.sql(f"""
    SELECT accession, harmonized_name, content
    FROM kbase_ke_pangenome.ncbi_env
    WHERE harmonized_name IN ({",".join(repr(f) for f in TARGET_FIELDS)})
      AND content IS NOT NULL
      AND content != ''
""")
env_wide = (
    env_long.groupBy("accession")
    .pivot("harmonized_name", TARGET_FIELDS)
    .agg(F.first("content"))
)
print(f"  Distinct biosamples with at least one env field: {env_wide.count():,}")


Step 2: Pivoting ncbi_env...


  Distinct biosamples with at least one env field: 279,938


In [7]:
# -----------------------------------------------------------------------------
# Step 3: AlphaEarth presence flag
# -----------------------------------------------------------------------------
print("\nStep 3: AlphaEarth presence flag...")
ae = spark.sql("""
    SELECT DISTINCT genome_id, 1 AS has_alphaearth
    FROM kbase_ke_pangenome.alphaearth_embeddings_all_years
""")
print(f"  Genomes with AlphaEarth: {ae.count():,}")


Step 3: AlphaEarth presence flag...


  Genomes with AlphaEarth: 83,227


In [8]:
# -----------------------------------------------------------------------------
# Step 4: Join everything
# -----------------------------------------------------------------------------
print("\nStep 4: Joining...")
master = (
    core
    .join(env_wide, core.ncbi_biosample_id == env_wide.accession, "left")
    .drop("accession")
    .join(ae, "genome_id", "left")
    .fillna(0, subset=["has_alphaearth"])
)


Step 4: Joining...


In [9]:
# Apply quality filter: keep all rows for now, flag quality so we can subset later
master = master.withColumn(
    "passes_quality",
    (F.col("checkm_completeness") >= 90) & (F.col("checkm_contamination") <= 5)
)

In [10]:
# -----------------------------------------------------------------------------
# Step 5: Collect to pandas (small — 293K rows × ~15 cols) and save
# -----------------------------------------------------------------------------
print("\nStep 5: Collecting to pandas and saving parquet...")
df_raw = master.toPandas()
# Clean copy to drop Spark Connect plan metadata that breaks pyarrow serialization
df = pd.DataFrame({c: df_raw[c].to_numpy() for c in df_raw.columns})
del df_raw
print(f"  Rows: {len(df):,}, cols: {len(df.columns)}")
print(f"  Columns: {list(df.columns)}")
out_path = os.path.join(DATA_DIR, "genome_gc_env.parquet")
df.to_parquet(out_path, index=False)
print(f"  Wrote {out_path}")


Step 5: Collecting to pandas and saving parquet...


  Rows: 293,059, cols: 18
  Columns: ['genome_id', 'gtdb_species_clade_id', 'ncbi_biosample_id', 'gc_pct', 'genome_size', 'checkm_completeness', 'checkm_contamination', 'isolation_source', 'host', 'env_broad_scale', 'env_local_scale', 'env_medium', 'lat_lon', 'geo_loc_name', 'sample_type', 'host_disease', 'has_alphaearth', 'passes_quality']
  Wrote /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/data/genome_gc_env.parquet


In [11]:
# -----------------------------------------------------------------------------
# Step 6: Summary statistics
# -----------------------------------------------------------------------------
print("\n=== Coverage summary ===")
summary = pd.DataFrame({
    "field": ["gc_pct", "genome_size", "checkm_completeness",
              "isolation_source", "host", "env_broad_scale",
              "env_local_scale", "env_medium", "lat_lon",
              "geo_loc_name", "host_disease", "has_alphaearth", "passes_quality"],
})
summary["n_nonnull"] = summary["field"].apply(
    lambda f: int(df[f].notna().sum()) if f != "has_alphaearth"
              else int((df[f] == 1).sum())
)
summary["pct"] = (100 * summary["n_nonnull"] / len(df)).round(1)
print(summary.to_string(index=False))
summary.to_csv(os.path.join(DATA_DIR, "01_field_coverage.csv"), index=False)


=== Coverage summary ===
              field  n_nonnull   pct
             gc_pct     293059 100.0
        genome_size     293059 100.0
checkm_completeness     293059 100.0
   isolation_source     245717  83.8
               host     170851  58.3
    env_broad_scale      88321  30.1
    env_local_scale      79863  27.3
         env_medium      79679  27.2
            lat_lon     137640  47.0
       geo_loc_name     271162  92.5
       host_disease      52842  18.0
     has_alphaearth      83227  28.4
     passes_quality     293059 100.0


In [12]:
# Species with most genomes
print("\n=== Top 20 species by genome count (quality-filtered) ===")
top_species = (
    df[df["passes_quality"] & df["gc_pct"].notna()]
    .groupby("gtdb_species_clade_id")
    .size()
    .sort_values(ascending=False)
    .head(20)
)
print(top_species.to_string())
top_species.to_csv(os.path.join(DATA_DIR, "01_top_species.csv"))


=== Top 20 species by genome count (quality-filtered) ===


gtdb_species_clade_id
s__Staphylococcus_aureus--RS_GCF_001027105.1         14504
s__Klebsiella_pneumoniae--RS_GCF_000742135.1         14176
s__Salmonella_enterica--RS_GCF_000006945.2           11373
s__Streptococcus_pneumoniae--RS_GCF_001457635.1       8363
s__Mycobacterium_tuberculosis--RS_GCF_000195955.2     6883
s__Pseudomonas_aeruginosa--RS_GCF_001457615.1         6737
s__Acinetobacter_baumannii--RS_GCF_009759685.1        6568
s__Clostridioides_difficile--RS_GCF_001077535.1       2585
s__Enterococcus_B_faecium--RS_GCF_001544255.1         2525
s__Enterobacter_hormaechei_A--RS_GCF_001729745.1      2435
s__Campylobacter_D_jejuni--RS_GCF_001457695.1         2292
s__Enterococcus_faecalis--RS_GCF_000392875.1          2241
s__Listeria_monocytogenes--RS_GCF_900187225.1         2199
s__Streptococcus_pyogenes--RS_GCF_002055535.1         2079
s__Neisseria_meningitidis--RS_GCF_900638555.1         2078
s__Listeria_monocytogenes_B--RS_GCF_000307025.1       1922
s__Mycobacterium_abscessus--RS_GCF

In [13]:
# -----------------------------------------------------------------------------
# Step 7: Figures
# -----------------------------------------------------------------------------
print("\nStep 7: Figures...")
qual = df[df["passes_quality"] & df["gc_pct"].notna()]


Step 7: Figures...


In [14]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(qual["gc_pct"], bins=80, color="#4c72b0", edgecolor="white")
ax.set_xlabel("GC content (%)")
ax.set_ylabel("Number of genomes")
ax.set_title(f"GC content distribution across {len(qual):,} quality-filtered bacterial genomes")
ax.axvline(qual["gc_pct"].mean(), color="black", linestyle="--", linewidth=1,
           label=f"mean = {qual['gc_pct'].mean():.1f}%")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_gc_distribution_overall.png"), dpi=150)
plt.close()

In [15]:
fig, ax = plt.subplots(figsize=(9, 5))
summary_plot = summary.set_index("field")["pct"].sort_values(ascending=True)
ax.barh(summary_plot.index, summary_plot.values, color="#55a868")
ax.set_xlabel("% of genomes with this field populated")
ax.set_title("Field coverage in the master genome table (n={:,})".format(len(df)))
for i, v in enumerate(summary_plot.values):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_field_coverage.png"), dpi=150)
plt.close()

In [16]:
print("\nNotebook 01 complete.")
print(f"Outputs:")
print(f"  {out_path}")
print(f"  {os.path.join(DATA_DIR, '01_field_coverage.csv')}")
print(f"  {os.path.join(DATA_DIR, '01_top_species.csv')}")
print(f"  {os.path.join(FIG_DIR, '01_gc_distribution_overall.png')}")
print(f"  {os.path.join(FIG_DIR, '01_field_coverage.png')}")


Notebook 01 complete.
Outputs:
  /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/data/genome_gc_env.parquet
  /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/data/01_field_coverage.csv
  /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/data/01_top_species.csv
  /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/figures/01_gc_distribution_overall.png
  /home/justaddcoffee/BERIL-research-observatory/projects/gc_ecotype_ecology/figures/01_field_coverage.png
